In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


In [2]:
CROP = 'corn'
PROJECT_ROOT = Path('.').resolve().parent                      # .. project root
INPUT_DIR = PROJECT_ROOT / 'data' / 'raw' / f'{CROP}'         # e.g., ../data/raw/cotton/
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / f'{CROP}'  # e.g., ../data/processed/cotton/

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if CROP == 'corn':
    XLS_SEARCH_TERM = 'Feed Grain and Corn Supply and Use'
elif CROP == 'cotton':
    XLS_SEARCH_TERM = 'Cotton Supply and Use'
elif CROP == 'soybeans':
    XLS_SEARCH_TERM = 'Soybeans and Products Supply and Use'
else:
    raise ValueError("`CROP` must be one of ['corn', 'cotton', 'soybeans']")


In [3]:
def convert_and_pad_list(input_list):
    """
    Given an input list, replace all string values with float equivalents.
    Then, make sure the list has 4 elements. If not, insert a NaN value at index 3.
    """
    input_list = [float(val) if val != 'na' else np.nan for val in input_list]
    if len(input_list) == 3:
        input_list.insert(2, np.nan)

    return input_list


In [4]:
def create_wasde_df(dates: list, plines: list, hlines: list, ylines: list) -> pandas.DataFrame:
    """
    Given an input list for dates, planted and harvested area data, and yield data,
    return a Pandas DataFrame representing scraped monthly data from the USDA WASDE site.

    This method works to build the DataFrame for manually created data and parsed TXT and XLS files.
    """
    df = pd.DataFrame({
        # duplicate time column for a datetime index that will be dropped later
        'dates': dates,
        'time': dates,
        
        'year_minus_2_planted_area': [num[0] for num in plines],
        'year_minus_1_planted_area': [num[1] for num in plines],
        'current_year_last_month_planted_area': [num[2] for num in plines],
        'current_year_current_month_planted_area': [num[3] for num in plines],

        'year_minus_2_harvested_area': [num[0] for num in hlines],
        'year_minus_1_harvested_area': [num[1] for num in hlines],
        'current_year_last_month_harvested_area': [num[2] for num in hlines],
        'current_year_current_month_harvested_area': [num[3] for num in hlines],

        'year_minus_2_yield': [num[0] for num in ylines],
        'year_minus_1_yield': [num[1] for num in ylines],
        'current_year_last_month_yield': [num[2] for num in ylines],
        'current_year_current_month_yield': [num[3] for num in ylines],
    })

    df = df.set_index('dates')

    return df


## Add some data in manually

We would like to have monthly data that goes as far back as possible. Unfortunately, before 1995, the data for some crops is in PDF form instead of TXT (and later, XLS) files. The data format also changes quite a bit over the years, both in naming conventions, where data is placed in a report (which interferes with regex search patterns), and more.

For now, we are going to add in some data manually, primarily the May data for 1991 - 1995. The official data the USDA uses in reports and figures comes out in May, so instead of adding in monthly data for these dates, we are only going to focus on annual data.


In [5]:
# https://esmis.nal.usda.gov/publication/world-agricultural-supply-and-demand-estimates?date=YYYY-05
# where YYYY could be: [1991, 1995]

if CROP == 'corn':
    dates = [pd.to_datetime(date) for date in ['1991-05-01', '1992-05-01', '1993-05-01', '1994-05-01']]

    pa_lines = [
        [72.2, 74.2, np.nan, np.nan],
        [74.2, 76.0, np.nan, np.nan],
        [76.0, 79.3, np.nan, 76.5],
        [79.3, 73.3, np.nan, 78.6],
    ]
    
    ha_lines = [
        [64.7, 67.0, np.nan, np.nan],
        [67.0, 68.8, np.nan, np.nan],
        [68.8, 72.1, np.nan, 69.3],
        [72.2, 63.0, np.nan, 71.5],
    ]
    
    y_lines = [
        [116.3, 118.5, np.nan, np.nan],
        [118.5, 108.6, np.nan, np.nan],
        [108.6, 131.4, np.nan, 122.7],
        [131.4, 100.7, np.nan, 122.1],
    ]

elif CROP == 'cotton':
    dates = [pd.to_datetime(date) for date in ['1991-05-01', '1992-05-01', '1993-05-01', '1994-05-01', '1995-05-01']]
    
    pa_lines = [
        [10.59, 12.35, np.nan, np.nan],
        [12.35, 14.05, np.nan, np.nan],
        [14.05, 13.24, np.nan, 13.43],
        [13.24, 13.44, np.nan, 13.84],
        [13.44, 13.73, np.nan, 16.20],
    ]
    
    ha_lines = [
        [9.54, 11.73, np.nan, np.nan],
        [11.73, 12.96, np.nan, np.nan],
        [12.96, 11.14, np.nan, 12.36],
        [11.14, 12.79, np.nan, 12.80],
        [12.78, 13.33, np.nan, 15.15],
    ]
    
    y_lines = [
        [614, 634, np.nan, np.nan],
        [634, 652, np.nan, np.nan],
        [652, 699, np.nan, 680],
        [699, 606, np.nan, 665],
        [606, 708, np.nan, 665],
    ]

else: # CROP == 'soybeans':
    dates = [pd.to_datetime(date) for date in ['1991-05-01', '1992-05-01', '1993-05-01', '1994-05-01']]

    pa_lines = [
        [60.8, 57.8, np.nan, np.nan],
        [57.8, 59.1, np.nan, np.nan],
        [59.2, 59.3, np.nan, 59.3],
        [59.1, 59.4, np.nan, 61.1],
    ]
    
    ha_lines = [
        [59.5, 56.5, np.nan, np.nan],
        [56.5, 58.0, np.nan, np.nan],
        [58.0, 58.4, np.nan, 59.3],
        [58.2, 56.4, np.nan, 60.0],
    ]
    
    y_lines = [
        [32.3, 34.0, np.nan, np.nan],
        [34.0, 34.3, np.nan, np.nan],
        [34.2, 37.6, np.nan, 35.1],
        [37.6, 32.0, np.nan, 35.0],
    ]

df_manual = create_wasde_df(dates, pa_lines, ha_lines, y_lines)
df_manual


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
dates,,,,,,,,,,,,,
1991-05-01,1991-05-01,72.2,74.2,NaN,NaN,64.7,67.0,NaN,NaN,116.3,118.5,NaN,NaN
1992-05-01,1992-05-01,74.2,76.0,NaN,NaN,67.0,68.8,NaN,NaN,118.5,108.6,NaN,NaN
1993-05-01,1993-05-01,76.0,79.3,NaN,76.5,68.8,72.1,NaN,69.3,108.6,131.4,NaN,122.7
1994-05-01,1994-05-01,79.3,73.3,NaN,78.6,72.2,63.0,NaN,71.5,131.4,100.7,NaN,122.1


## Read in and process text files

In [6]:
txt_filenames = sorted(list(INPUT_DIR.glob('*.txt')))
for filename in txt_filenames[:10]:
    filename = str(filename)
    start_index = filename.index('usda-wasde-web-scraping')
    print(filename[start_index:])
print()
for filename in txt_filenames[-10:]:
    filename = str(filename)
    start_index = filename.index('usda-wasde-web-scraping')
    print(filename[start_index:])


usda-wasde-web-scraping/data/raw/corn/corn_1995_01.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_02.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_03.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_04.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_05.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_06.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_07.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_08.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_09.txt
usda-wasde-web-scraping/data/raw/corn/corn_1995_10.txt

usda-wasde-web-scraping/data/raw/corn/corn_2025_05.txt
usda-wasde-web-scraping/data/raw/corn/corn_2025_06.txt
usda-wasde-web-scraping/data/raw/corn/corn_2025_07.txt
usda-wasde-web-scraping/data/raw/corn/corn_2025_08.txt
usda-wasde-web-scraping/data/raw/corn/corn_2025_09.txt
usda-wasde-web-scraping/data/raw/corn/corn_2025_11.txt
usda-wasde-web-scraping/data/raw/corn/corn_2025_12.txt
usda-wasde-web-scraping/data/raw/corn/corn_2026_01.txt
usda-wasd

In [7]:
dates = []
pa_lines = []
ha_lines = []
y_lines = []

for filename in txt_filenames:
    # convert PosixPath to str
    filename = str(filename)
    
    year, month = filename[-11:-4].split('_')
    date = f"{year}-{month}-01"
    dates.append(pd.to_datetime(date))
    
    with open(filename) as f:
        lines = [line.lower().replace('\n', '').replace('*', '').replace(':', '').lstrip().rstrip() for line in f]

        if CROP == 'corn':
            matching_index = lines.index('corn')
            planted_index = matching_index + 2 if pd.to_datetime(date).year < 2016 else matching_index + 1
            harvested_index = matching_index + 3 if pd.to_datetime(date).year < 2016 else matching_index + 2
            # works for both pre- and post-2016 data
            yield_index = matching_index + 5
            
            matching_lines = lines[planted_index]
            pa_line = matching_lines.replace('area', '').replace('planted', '').split()
            pa_line = convert_and_pad_list(pa_line)
            pa_lines.append(pa_line)
    
            matching_lines = lines[harvested_index]
            ha_line = matching_lines.replace('area', '').replace('harvested', '').split()
            ha_line = convert_and_pad_list(ha_line)
            ha_lines.append(ha_line)
            
            matching_lines = lines[yield_index]
            # the 2016 data format change includes the word 'acre' in the line we are looking for
            # we can apply the .replace() method to all lines because if the term isn't found, we don't replace anything
            y_line = matching_lines.replace('yield per harvested', '').replace('acre', '').split()
            y_line = convert_and_pad_list(y_line)
            y_lines.append(y_line)

        elif CROP == 'cotton':
            matching_lines = [line for line in lines if 'planted' in line]
            pa_line = matching_lines[0].replace('planted', '').split()
            pa_line = convert_and_pad_list(pa_line)
            pa_lines.append(pa_line)
    
            matching_lines = [line for line in lines if 'harvested' in line]
            ha_line = matching_lines[0].replace('harvested', '').split()
            ha_line = convert_and_pad_list(ha_line)
            ha_lines.append(ha_line)
        
            # the label for the yield row changes in June 1998 
            if pd.to_datetime(f'{year}-{month}-01') < pd.to_datetime('1998-06-01'):
                matching_lines = [line for line in lines if 'yield per harv. acre' in line]
                y_line = matching_lines[0].replace('yield per harv. acre', '').split()
            else:
                pattern = re.compile(r'^.*\bacre\b.*$', re.MULTILINE | re.IGNORECASE)
                matching_lines = [line for line in lines if pattern.search(line)]
                y_line = matching_lines[0].replace('acre', '').split()

            y_line = convert_and_pad_list(y_line)
            y_lines.append(y_line)
        
        else: # CROP == 'soybeans'
            matching_lines = [line for line in lines if 'planted' in line]
            pa_line = matching_lines[0].replace('area', '').replace('planted', '').split()
            pa_line = convert_and_pad_list(pa_line)
            pa_lines.append(pa_line)
    
            matching_lines = [line for line in lines if 'harvested' in line]
            ha_line = matching_lines[0].replace('area', '').replace('harvested', '').split()
            ha_line = convert_and_pad_list(ha_line)
            ha_lines.append(ha_line)
        
            # the label for the yield row changes in June 1998 
            if pd.to_datetime(f'{year}-{month}-01') < pd.to_datetime('1996-05-01'):
                matching_lines = [line for line in lines if 'unit' in line]
                y_line = matching_lines[0].replace('unit', '').split()
            elif pd.to_datetime(f'{year}-{month}-01') == pd.to_datetime('1996-05-01'):
                matching_lines = [line for line in lines if 'yield per harv. acre' in line]
                y_line = matching_lines[0].replace('yield per harv. acre', '').split()
            elif pd.to_datetime(f'{year}-{month}-01') > pd.to_datetime('1996-05-01') and pd.to_datetime(f'{year}-{month}-01') < pd.to_datetime('2016-10-01'):
                pattern = re.compile(r'^.*\bacre\b.*$', re.MULTILINE | re.IGNORECASE)
                matching_lines = [line for line in lines if pattern.search(line)]
                y_line = matching_lines[0].replace('acre', '').split()
            else: # date > 2016-10-01
                matching_lines = [line for line in lines if 'yield per harvested acre' in line]
                y_line = matching_lines[0].replace('yield per harvested acre', '').split()

            y_line = convert_and_pad_list(y_line)
            y_lines.append(y_line)

print(len(dates))
print(len(pa_lines))
print(len(ha_lines))
print(len(y_lines))


301
301
301
301


In [8]:
df_txt = create_wasde_df(dates, pa_lines, ha_lines, y_lines)
df_txt


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
dates,,,,,,,,,,,,,
1995-01-01,1995-01-01,79.3,73.2,79.1,79.2,72.1,62.9,72.3,72.9,131.5,100.7,138.4,138.6
1995-02-01,1995-02-01,79.3,73.2,79.2,79.2,72.1,62.9,72.9,72.9,131.5,100.7,138.6,138.6
1995-03-01,1995-03-01,79.3,73.2,79.2,79.2,72.1,62.9,72.9,72.9,131.5,100.7,138.6,138.6
1995-04-01,1995-04-01,79.3,73.2,79.2,79.2,72.1,62.9,72.9,72.9,131.5,100.7,138.6,138.6
1995-05-01,1995-05-01,73.2,79.2,NaN,75.3,62.9,72.9,NaN,68.5,100.7,138.6,NaN,125.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-01,2025-11-01,94.6,90.9,98.7,98.7,86.5,83.0,90.0,90.0,177.3,179.3,186.7,186.0
2025-12-01,2025-12-01,94.6,90.9,98.7,98.7,86.5,83.0,90.0,90.0,177.3,179.3,186.0,186.0
2026-01-01,2026-01-01,94.6,90.9,98.7,98.8,86.5,83.0,90.0,91.3,177.3,179.3,186.0,186.5


In [9]:
# print missing dates before having processed xls files
missing_dates = pd.date_range(start=df_txt.index.min(), end=df_txt.index.max(), freq='MS').difference(df_txt.index).tolist()
missing_dates = [date.strftime('%Y-%m-%d') for date in missing_dates]
print(missing_dates)


['2010-10-01', '2010-11-01', '2010-12-01', '2011-01-01', '2011-02-01', '2011-03-01', '2011-04-01', '2011-05-01', '2011-06-01', '2011-07-01', '2011-08-01', '2011-09-01', '2011-10-01', '2011-11-01', '2011-12-01', '2012-01-01', '2012-02-01', '2012-03-01', '2012-04-01', '2012-05-01', '2012-06-01', '2012-07-01', '2012-08-01', '2012-09-01', '2012-10-01', '2012-11-01', '2012-12-01', '2013-01-01', '2013-02-01', '2013-03-01', '2013-04-01', '2013-05-01', '2013-06-01', '2013-07-01', '2013-08-01', '2013-09-01', '2013-10-01', '2013-11-01', '2013-12-01', '2014-01-01', '2014-02-01', '2014-03-01', '2014-04-01', '2014-05-01', '2014-06-01', '2014-07-01', '2014-08-01', '2014-09-01', '2014-10-01', '2014-11-01', '2014-12-01', '2015-01-01', '2015-02-01', '2015-03-01', '2015-04-01', '2015-05-01', '2015-06-01', '2015-07-01', '2015-08-01', '2015-09-01', '2015-10-01', '2015-11-01', '2015-12-01', '2016-01-01', '2016-02-01', '2016-03-01', '2016-04-01', '2016-05-01', '2016-06-01', '2016-07-01', '2016-08-01', '2016

## Read in and process XLS files

In [10]:
xls_filenames = sorted(list(INPUT_DIR.glob('*.xls')))
for filename in xls_filenames[:10]:
    filename = str(filename)
    start_index = filename.index('usda-wasde-web-scraping')
    print(filename[start_index:])
print()
for filename in xls_filenames[-10:]:
    filename = str(filename)
    start_index = filename.index('usda-wasde-web-scraping')
    print(filename[start_index:])


usda-wasde-web-scraping/data/raw/corn/corn_2010_10.xls
usda-wasde-web-scraping/data/raw/corn/corn_2010_11.xls
usda-wasde-web-scraping/data/raw/corn/corn_2010_12.xls
usda-wasde-web-scraping/data/raw/corn/corn_2011_01.xls
usda-wasde-web-scraping/data/raw/corn/corn_2011_02.xls
usda-wasde-web-scraping/data/raw/corn/corn_2011_03.xls
usda-wasde-web-scraping/data/raw/corn/corn_2011_04.xls
usda-wasde-web-scraping/data/raw/corn/corn_2011_05.xls
usda-wasde-web-scraping/data/raw/corn/corn_2011_06.xls
usda-wasde-web-scraping/data/raw/corn/corn_2011_07.xls

usda-wasde-web-scraping/data/raw/corn/corn_2015_12.xls
usda-wasde-web-scraping/data/raw/corn/corn_2016_01.xls
usda-wasde-web-scraping/data/raw/corn/corn_2016_02.xls
usda-wasde-web-scraping/data/raw/corn/corn_2016_03.xls
usda-wasde-web-scraping/data/raw/corn/corn_2016_04.xls
usda-wasde-web-scraping/data/raw/corn/corn_2016_05.xls
usda-wasde-web-scraping/data/raw/corn/corn_2016_06.xls
usda-wasde-web-scraping/data/raw/corn/corn_2016_07.xls
usda-wasd

In [11]:
dates = []
pa_lines = []
ha_lines = []
y_lines = []

for filename in xls_filenames:
    # convert PosixPath to str
    filename = str(filename)
    
    year, month = filename[-11:-4].split('_')
    date = f"{year}-{month}-01"
    dates.append(pd.to_datetime(date))

    # test case files: cotton_2016_08 has a different format than cotton_2010_10
    df = pd.read_excel(filename)
    # drop filler columns and rows
    df = df[~df.apply(lambda row: row.astype(str).str.contains(XLS_SEARCH_TERM).any(), axis=1)]
    df = df[~df.apply(lambda row: row.astype(str).str.contains('WASDE').any(), axis=1)]
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)

    if CROP == 'corn':
        df = df.reset_index(drop=True)

        # we want to ignore everything between the lines `FEED GRAINS` and `CORN`
        # since we are only interested in planted / harvested acres and yield info for corn
        start_index = df.iloc[:, 0].values.tolist().index(f'{CROP.upper()}')
        df = df.iloc[start_index:, :]
    
        df.columns = ['label', 'year_minus_2', 'year_minus_1', 'last_month', 'current_month']
        df['label'] = df.label.str.lower().str.strip()
        df = df.query(" label == 'area planted' or label == 'area harvested' or label == 'yield per harvested acre' ")
        
        # remove all asterisks
        for column in df.columns:
            df[column] = df[column].astype(str).str.replace('*', '')
    
        pa_line = df.query(" label == 'area planted' ").iloc[:, 1:].values[0].tolist()
        ha_line = df.query(" label == 'area harvested' ").iloc[:, 1:].values[0].tolist()
        y_line = df.query(" label == 'yield per harvested acre' ").iloc[:, 1:].values[0].tolist()
        
        pa_line = convert_and_pad_list(pa_line)
        ha_line = convert_and_pad_list(ha_line)
        y_line = convert_and_pad_list(y_line)
    
        pa_lines.append(pa_line)
        ha_lines.append(ha_line)
        y_lines.append(y_line)

    elif CROP == 'cotton':
        df.columns = ['label', 'year_minus_2', 'year_minus_1', 'last_month', 'current_month']
    
        df['label'] = df.label.str.lower().str.strip()
        df = df.query(" label == 'planted' or label == 'harvested' or label == 'yield per harvested acre' ")
    
        # remove all asterisks
        for column in df.columns:
            df[column] = df[column].astype(str).str.replace('*', '')
        
        pa_line = df.query(" label == 'planted' ").iloc[:, 1:].values[0].tolist()
        ha_line = df.query(" label == 'harvested' ").iloc[:, 1:].values[0].tolist()
        y_line = df.query(" label == 'yield per harvested acre' ").iloc[:, 1:].values[0].tolist()
        
        pa_line = convert_and_pad_list(pa_line)
        ha_line = convert_and_pad_list(ha_line)
        y_line = convert_and_pad_list(y_line)
    
        pa_lines.append(pa_line)
        ha_lines.append(ha_line)
        y_lines.append(y_line)

    else: # CROP == 'soybeans'
        if pd.to_datetime(date) < pd.to_datetime('2015-07-01'):
            df.columns = ['label', 'date', 'year_minus_2', 'year_minus_1', 'last_month', 'current_month']
        else:
            df.columns = ['label', 'year_minus_2', 'year_minus_1', 'last_month', 'current_month']

        df['label'] = df.label.str.lower().str.strip()
        df = df.query(" label == 'area planted' or label == 'area harvested' or label == 'yield per harvested acre' ")
        
        # remove all asterisks
        for column in df.columns:
            df[column] = df[column].astype(str).str.replace('*', '')
    
        # the soybean data has an extra 'date' column for some month-years, so we need to shift up a column in the values that we take
        if pd.to_datetime(date) < pd.to_datetime('2015-07-01'):
            pa_line = df.query(" label == 'area planted' ").iloc[:, 2:].values[0].tolist()
            ha_line = df.query(" label == 'area harvested' ").iloc[:, 2:].values[0].tolist()
            y_line = df.query(" label == 'yield per harvested acre' ").iloc[:, 2:].values[0].tolist()
        else:
            pa_line = df.query(" label == 'area planted' ").iloc[:, 1:].values[0].tolist()
            ha_line = df.query(" label == 'area harvested' ").iloc[:, 1:].values[0].tolist()
            y_line = df.query(" label == 'yield per harvested acre' ").iloc[:, 1:].values[0].tolist()
        
        pa_line = convert_and_pad_list(pa_line)
        ha_line = convert_and_pad_list(ha_line)
        y_line = convert_and_pad_list(y_line)
    
        pa_lines.append(pa_line)
        ha_lines.append(ha_line)
        y_lines.append(y_line)

print(len(dates))
print(len(pa_lines))
print(len(ha_lines))
print(len(y_lines))


71
71
71
71


In [12]:
df_xls = create_wasde_df(dates, pa_lines, ha_lines, y_lines)
df_xls


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
dates,,,,,,,,,,,,,
2010-10-01,2010-10-01,86.0,86.5,87.9,88.2,78.6,79.6,81.0,81.3,153.9,164.7,162.5,155.8
2010-11-01,2010-11-01,86.0,86.5,88.2,88.2,78.6,79.6,81.3,81.3,153.9,164.7,155.8,154.3
2010-12-01,2010-12-01,86.0,86.5,88.2,88.2,78.6,79.6,81.3,81.3,153.9,164.7,154.3,154.3
2011-01-01,2011-01-01,86.0,86.4,88.2,88.2,78.6,79.5,81.3,81.4,153.9,164.7,154.3,152.8
2011-02-01,2011-02-01,86.0,86.4,88.2,88.2,78.6,79.5,81.4,81.4,153.9,164.7,152.8,152.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-05-01,2016-05-01,90.6,88.0,NaN,93.6,83.1,80.7,NaN,85.9,171.0,168.4,NaN,168.0
2016-06-01,2016-06-01,90.6,88.0,93.6,93.6,83.1,80.7,85.9,85.9,171.0,168.4,168.0,168.0
2016-07-01,2016-07-01,90.6,88.0,93.6,94.1,83.1,80.7,85.9,86.6,171.0,168.4,168.0,168.0


## Merge processed dataframes for text and XLS files

In [13]:
# df_txt already has empty rows for the dates where there are xls files
# so we want to infill those while keeping df_txt the same
df = pd.concat([df_manual, df_txt, df_xls]).sort_values(by='time')
df


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
dates,,,,,,,,,,,,,
1991-05-01,1991-05-01,72.2,74.2,NaN,NaN,64.7,67.0,NaN,NaN,116.3,118.5,NaN,NaN
1992-05-01,1992-05-01,74.2,76.0,NaN,NaN,67.0,68.8,NaN,NaN,118.5,108.6,NaN,NaN
1993-05-01,1993-05-01,76.0,79.3,NaN,76.5,68.8,72.1,NaN,69.3,108.6,131.4,NaN,122.7
1994-05-01,1994-05-01,79.3,73.3,NaN,78.6,72.2,63.0,NaN,71.5,131.4,100.7,NaN,122.1
1995-01-01,1995-01-01,79.3,73.2,79.1,79.2,72.1,62.9,72.3,72.9,131.5,100.7,138.4,138.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-01,2025-11-01,94.6,90.9,98.7,98.7,86.5,83.0,90.0,90.0,177.3,179.3,186.7,186.0
2025-12-01,2025-12-01,94.6,90.9,98.7,98.7,86.5,83.0,90.0,90.0,177.3,179.3,186.0,186.0
2026-01-01,2026-01-01,94.6,90.9,98.7,98.8,86.5,83.0,90.0,91.3,177.3,179.3,186.0,186.5


In [14]:
full_date_list = pd.date_range(start=df.index.min(), end=df.index.max(), freq='MS')
df = df.reindex(full_date_list)
df['time'] = df.index

# now let's format the time column as a string
df = df.reset_index(drop=True)
df['time'] = df.time.dt.strftime('%Y-%m-%d')

df


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
0,1991-05-01,72.2,74.2,NaN,NaN,64.7,67.0,NaN,NaN,116.3,118.5,NaN,NaN
1,1991-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1991-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1991-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1991-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,2025-11-01,94.6,90.9,98.7,98.7,86.5,83.0,90.0,90.0,177.3,179.3,186.7,186.0
415,2025-12-01,94.6,90.9,98.7,98.7,86.5,83.0,90.0,90.0,177.3,179.3,186.0,186.0
416,2026-01-01,94.6,90.9,98.7,98.8,86.5,83.0,90.0,91.3,177.3,179.3,186.0,186.5
417,2026-02-01,94.6,90.9,98.8,98.8,86.5,83.0,91.3,91.3,177.3,179.3,186.5,186.5


In [15]:
# check for missing dates
data_cols = df.columns.values[1:].tolist()
all_nan_mask = df[data_cols].isna().all(axis=1)
found_missing_dates = df.loc[all_nan_mask, 'time'].tolist()
print(found_missing_dates)

known_missing_dates = ['2013-10-01', '2019-01-01', '2025-10-01']
# if this test passes, then we have data for all but when we know it is missing
# because we only imported data for may in 1992-95, we can ignore those dates
assert(found_missing_dates[-3:] == known_missing_dates)


['1991-06-01', '1991-07-01', '1991-08-01', '1991-09-01', '1991-10-01', '1991-11-01', '1991-12-01', '1992-01-01', '1992-02-01', '1992-03-01', '1992-04-01', '1992-06-01', '1992-07-01', '1992-08-01', '1992-09-01', '1992-10-01', '1992-11-01', '1992-12-01', '1993-01-01', '1993-02-01', '1993-03-01', '1993-04-01', '1993-06-01', '1993-07-01', '1993-08-01', '1993-09-01', '1993-10-01', '1993-11-01', '1993-12-01', '1994-01-01', '1994-02-01', '1994-03-01', '1994-04-01', '1994-06-01', '1994-07-01', '1994-08-01', '1994-09-01', '1994-10-01', '1994-11-01', '1994-12-01', '2013-10-01', '2019-01-01', '2025-10-01']


In [16]:
# let's make sure that xls data made it into the main dataframe
df.query(" @pd.to_datetime(time) >= @df_xls.time.min() and @pd.to_datetime(time) <= @df_xls.time.max() ").head(25)


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
233,2010-10-01,86.0,86.5,87.9,88.2,78.6,79.6,81.0,81.3,153.9,164.7,162.5,155.8
234,2010-11-01,86.0,86.5,88.2,88.2,78.6,79.6,81.3,81.3,153.9,164.7,155.8,154.3
235,2010-12-01,86.0,86.5,88.2,88.2,78.6,79.6,81.3,81.3,153.9,164.7,154.3,154.3
236,2011-01-01,86.0,86.4,88.2,88.2,78.6,79.5,81.3,81.4,153.9,164.7,154.3,152.8
237,2011-02-01,86.0,86.4,88.2,88.2,78.6,79.5,81.4,81.4,153.9,164.7,152.8,152.8
238,2011-03-01,86.0,86.4,88.2,88.2,78.6,79.5,81.4,81.4,153.9,164.7,152.8,152.8
239,2011-04-01,86.0,86.4,88.2,88.2,78.6,79.5,81.4,81.4,153.9,164.7,152.8,152.8
240,2011-05-01,86.4,88.2,NaN,92.2,79.5,81.4,NaN,85.1,164.7,152.8,NaN,158.7
241,2011-06-01,86.4,88.2,92.2,90.7,79.5,81.4,85.1,83.2,164.7,152.8,158.7,158.7
242,2011-07-01,86.4,88.2,90.7,92.3,79.5,81.4,83.2,84.9,164.7,152.8,158.7,158.7


In [17]:
df.query(" time.isin(@known_missing_dates)  ")


,time,year_minus_2_planted_area,year_minus_1_planted_area,current_year_last_month_planted_area,current_year_current_month_planted_area,year_minus_2_harvested_area,year_minus_1_harvested_area,current_year_last_month_harvested_area,current_year_current_month_harvested_area,year_minus_2_yield,year_minus_1_yield,current_year_last_month_yield,current_year_current_month_yield
269,2013-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
332,2019-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
413,2025-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save output

In [18]:
df.to_excel(f'{OUTPUT_DIR}/{CROP}_1991_2026.xlsx', index=False)
